In [9]:
import numpy as np
import numpy.testing as npt

from scipy import stats
from scipy.stats import t,ttest_ind
from scipy.stats import f
from scipy.stats import f_oneway
from scipy.stats import pearsonr

import statsmodels.api as sm
from statsmodels.regression._prediction import get_prediction
from statsmodels.stats.outliers_influence import OLSInfluence,MLEInfluence
from statsmodels.graphics.gofplots import qqplot_2samples,ProbPlot,qqplot
import pandas as pd
from patsy import dmatrices
from numpy.testing import assert_almost_equal, assert_allclose
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns
import os

# some_file.py
import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, r'C:\Users\TODO\Desktop\Abhi\AI\AI\Math\Hands-On\statemodelsStudy')
from olsRegressionAnalysis import dispAnalysisOfVariance, tableDispFormatt,getInvOfProductMat,getRegressionEqn,\
                                  dispReghressionAnalysis,norm_scalling,getCorrelation,\
                                  get_variance_inflation_factors,getEigvals

In [10]:
path = os.path.join(os.getcwd(), 'DataSet', 'AcetyleneData_Stand_10_3.txt')
df = pd.read_csv(path)

# Step 1: Standrizations
'''
This dataset already standrized
'''

#Step 2: Calculate beta(R)
'''
Formula
beat(R) = (X'X + kI)^-1 X'y
'''

In [11]:
#Step 2.1 Prepare dataset according table 10.10
def getAugmentedMatrixXa(Exdox, Endox, k, yVar):    
    ser = pd.Series(Endox)
    zero = np.zeros((Exdox.to_numpy()).shape[1])
    ser2 = pd.Series(zero)
    dfY = pd.concat([ser, ser2])
    I = np.identity((Exdox.to_numpy()).shape[1])*np.sqrt(k)
    dfI = pd.DataFrame(I , columns = Exdox.columns.values)
    dfRidgereg = pd.concat([Exdox, dfI])
    dfRidgereg[yVar] = dfY
    return dfRidgereg

In [12]:
dfR = getAugmentedMatrixXa(Exdox = df.loc[:,['x1', 'x2', 'x3', 'x1x2', 'x1x3', 'x2x3', 'x1_Sqr', 'x2_Sqr','x3_Sqr']] , 
                     Endox=df.loc[:,'y'],
                     k = 0.032,
                     yVar = 'y'
                     )

In [13]:
# Step 3: apply simpaly OLS
formulas = 'y  ~  x1 + x2 + x3 + x1x2 + x1x3 + x2x3 + x1_Sqr + x2_Sqr + x3_Sqr'
y, X = dmatrices(
                 formula_like = formulas, 
                 data = dfR,
                 return_type = 'dataframe'                 
                 )

res = sm.OLS(y, X).fit()
print(res.params)

Intercept    0.083653
x1           0.180049
x2          -0.613973
x3          -0.733532
x1x2        -0.085381
x1x3        -0.693582
x2x3        -0.550367
x1_Sqr      -0.378341
x2_Sqr       0.284188
x3_Sqr      -0.005200
dtype: float64
